In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
books = pd.read_csv("../data/books_clean.csv")
ratings = pd.read_csv("../data/explicit_ratings.csv")

In [3]:
books.shape

(271360, 8)

In [4]:
ratings.shape

(383842, 3)

In [5]:
ratings_with_books = ratings.merge(
    books,
    on="ISBN",
    how="inner"
)

In [6]:
ratings_with_books.head()

,User_ID,ISBN,Book_Rating,Book_Title,Book_Author,Year_Of_Publication,Publisher,Image_URL_S,Image_URL_M,Image_URL_L
0,276726,0155061224,5,Rites of Passage,Judith Rae,2001.0,Heinle,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...,http://images.amazon.com/images/P/0155061224.0...
1,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999.0,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...,http://images.amazon.com/images/P/052165615X.0...
2,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001.0,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...,http://images.amazon.com/images/P/0521795028.0...
3,276744,038550120X,7,A Painted House,JOHN GRISHAM,2001.0,Doubleday,http://images.amazon.com/images/P/038550120X.0...,http://images.amazon.com/images/P/038550120X.0...,http://images.amazon.com/images/P/038550120X.0...
4,276747,0060517794,9,Little Altars Everywhere,Rebecca Wells,2003.0,HarperTorch,http://images.amazon.com/images/P/0060517794.0...,http://images.amazon.com/images/P/0060517794.0...,http://images.amazon.com/images/P/0060517794.0...


Explore User Activity

In [7]:
user_rating_count = ratings_with_books.groupby("User_ID").count()["Book_Rating"]

In [8]:
# active_users = user_rating_count[user_rating_count > 200].index

In [9]:
# Number of users remaining at different thresholds

for threshold in [5, 10, 20, 30, 50, 100, 200]:
    users = user_rating_count[user_rating_count >= threshold]
    print(f"{threshold:>3} ratings : {len(users)} users")

  5 ratings : 12787 users
 10 ratings : 6589 users
 20 ratings : 3305 users
 30 ratings : 2140 users
 50 ratings : 1180 users
100 ratings : 449 users
200 ratings : 118 users


Explore Book Popularity

In [10]:
book_rating_count = (
    ratings_with_books
    .groupby("Book_Title")["Book_Rating"]
    .count()
)

In [11]:
for threshold in [5, 10, 20, 30, 50]:
    books_count = book_rating_count[book_rating_count >= threshold]
    print(f"{threshold:>3} ratings : {len(books_count)} books")

  5 ratings : 13740 books
 10 ratings : 5712 books
 20 ratings : 2338 books
 30 ratings : 1324 books
 50 ratings : 651 books


Filtering

In [12]:
MIN_USER_RATINGS = 20
MIN_BOOK_RATINGS = 20

print(f"Minimum User Ratings : {MIN_USER_RATINGS}")
print(f"Minimum Book Ratings : {MIN_BOOK_RATINGS}")

Minimum User Ratings : 20
Minimum Book Ratings : 20


In [13]:
# Keep active users

active_users = user_rating_count[
    user_rating_count >= MIN_USER_RATINGS
].index

filtered_ratings = ratings_with_books[
    ratings_with_books["User_ID"].isin(active_users)
]

print("Filtered Ratings Shape:", filtered_ratings.shape)


Filtered Ratings Shape: (217729, 10)


In [14]:
# Count ratings again AFTER filtering users

book_rating_count = (
    filtered_ratings
    .groupby("Book_Title")["Book_Rating"]
    .count()
)

In [15]:
# Keep only popular books

popular_books = book_rating_count[
    book_rating_count >= MIN_BOOK_RATINGS
].index

final_ratings = filtered_ratings[
    filtered_ratings["Book_Title"].isin(popular_books)
]

print("Final Ratings Shape:", final_ratings.shape)

Final Ratings Shape: (38132, 10)


In [16]:
# Preview

final_ratings.head()

# %%
# Check remaining users and books

print("Unique Users :", final_ratings["User_ID"].nunique())
print("Unique Books :", final_ratings["Book_Title"].nunique())
print("Total Ratings:", len(final_ratings))

Unique Users : 3056
Unique Books : 973
Total Ratings: 38132


In [17]:
# Create pivot table
book_pivot = final_ratings.pivot_table(
    index="Book_Title",
    columns="User_ID",
    values="Book_Rating"
)

book_pivot.fillna(0, inplace=True)

book_pivot.shape

(973, 3056)

In [18]:
# Compute cosine similarity between books

similarity_scores = cosine_similarity(book_pivot)

print(similarity_scores.shape)

(973, 973)


In [19]:
def recommend(book_name, n=5):

    # Check if book exists
    if book_name not in book_pivot.index:
        return "Book not found."

    # Get index of the selected book
    index = np.where(book_pivot.index == book_name)[0][0]

    # Compute similarity scores
    similar_books = list(enumerate(similarity_scores[index]))

    # Sort by similarity
    similar_books = sorted(
        similar_books,
        key=lambda x: x[1],
        reverse=True
    )[1:n+1]

    recommendations = []

    for i in similar_books:

        title = book_pivot.index[i[0]]

        temp = books[
            books["Book_Title"] == title
        ].drop_duplicates("Book_Title")

        recommendations.append({
            "Book_Title": temp["Book_Title"].values[0],
            "Author": temp["Book_Author"].values[0],
            "Publisher": temp["Publisher"].values[0],
            "Similarity": round(i[1], 3),
            "Image_URL": temp["Image_URL_L"].values[0]
        })

    return pd.DataFrame(recommendations)

In [20]:
recommend("The Da Vinci Code")

,Book_Title,Author,Publisher,Similarity,Image_URL
0,Angels &amp; Demons,Dan Brown,Pocket Star,0.260,http://images.amazon.com/images/P/0671027360.0...
1,Middlesex: A Novel,Jeffrey Eugenides,Picador,0.189,http://images.amazon.com/images/P/0312422156.0...
2,Digital Fortress : A Thriller,Dan Brown,St. Martin's Press,0.172,http://images.amazon.com/images/P/0312995423.0...
3,The Lovely Bones: A Novel,Alice Sebold,"Little, Brown",0.170,http://images.amazon.com/images/P/0316666343.0...
4,The Secret Life of Bees,Sue Monk Kidd,Penguin Books,0.166,http://images.amazon.com/images/P/0142001740.0...


In [21]:
recommend("Harry Potter and the Chamber of Secrets")

'Book not found.'

In [23]:
book_pivot.index[
    book_pivot.index.str.contains("Harry", case=False)
]

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Order of the Phoenix (Book 5)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Sorcerer's Stone (Book 1)',
       'Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))'],
      dtype='str', name='Book_Title')

In [24]:
import pickle

# Save pivot table
book_pivot.to_pickle("../models/book_pivot.pkl")

# Save similarity matrix
with open("../models/similarity.pkl", "wb") as f:
    pickle.dump(similarity_scores, f)

# Save processed books
books.to_pickle("../models/books.pkl")

print("Models saved successfully!")

Models saved successfully!
